# Lab 07: Modern Machine Learning — Neural Networks and Image Classification
**Week 8 — November 2 · Due Monday, November 9 at 5:00 PM**

In this lab you will build a **multilayer perceptron (MLP)** to recognize handwritten digits from **0 to 9**. Each example is an image with a known digit label. You will prepare the pixel values, train a neural network, compare hidden-layer sizes, and examine its predictions on held-out images.

By the end, you should be able to:
- Prepare image pixels as numerical features for classification.
- Build an MLP and complete the main steps of its training loop.
- Use validation data to choose a hidden-layer size.
- Compare test accuracy with simple baselines and interpret classification errors.

Work from top to bottom. Run cells with **Shift + Enter**. Replace every **...** in an answer cell and complete the written responses. You can work with a neighbor and ask a tutor/TA for help.

Most of the training loop is provided. Complete Q1–Q7 before the optional sections on **embeddings, NLP, and transformers**.

## Setup

The **digits** dataset is included with scikit-learn. It contains **1,797 grayscale images**, each **8 × 8 pixels**, with labels from 0 to 9. This is a smaller dataset than MNIST, which uses 28 × 28 images. No separate data file or dataset download is needed.

Run the setup cell below. If PyTorch is missing, the cell installs it into your notebook’s Python environment. On DataHub, it downloads the CPU build. The first run may take a few minutes and requires internet access. If another library is missing, uncomment the optional installation line, run the cell, then restart the kernel and run the notebook from the top.

Public Otter checks cover Q1–Q6. Plots and written responses (including Q7) are reviewed separately. Keep the `tests/lab07` folder beside this notebook. Run each check after completing its exercise; a passing check is feedback on the code, not a grade for your written reasoning.


In [ ]:
# Uncomment only if another library is missing:
# %pip install numpy pandas matplotlib scikit-learn otter-grader

from importlib.util import find_spec
import subprocess
import sys

if find_spec("torch") is None:
    print("Installing PyTorch for this notebook kernel…", flush=True)
    install_torch = [sys.executable, "-m", "pip", "install", "--no-cache-dir", "torch>=2.2,<3"]
    if sys.platform == "linux":
        install_torch.extend(["--index-url", "https://download.pytorch.org/whl/cpu"])
    subprocess.run(install_torch, check=True)

import otter

grader = otter.Notebook("Lab07.ipynb", tests_dir="tests/lab07")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay
from sklearn.decomposition import PCA
from IPython.display import display

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)  # Small models run comfortably on a CPU.
plt.rcParams.update({'figure.figsize': (7, 4), 'axes.spines.top': False,
                     'axes.spines.right': False})
print('PyTorch:', torch.__version__)

## Part 1: Prepare the data

Each pixel has a value from **0** (dark) to **16** (bright). Each image has one integer label identifying the digit. Run the next cell to inspect the array shapes and view the first ten images.

In [ ]:
digits = load_digits()
images = digits.images
y = digits.target
print('Image array:', images.shape, '| Label array:', y.shape)
print('Pixel range:', images.min(), 'to', images.max())

fig, axes = plt.subplots(2, 5, figsize=(9, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i], cmap='gray', vmin=0, vmax=16)
    ax.set_title(f'Label: {y[i]}')
    ax.axis('off')
plt.tight_layout()
plt.show()

### Q1. Flatten and scale the images

Save a feature array named **X** with one row per image and **64 columns**, one for each pixel. Divide the pixel values by **16** so they range from 0 to 1. You should end up with shape **(1797, 64)**.

**Hint:** `images.reshape(len(images), -1)` flattens each image into a row. `-1` lets NumPy determine the number of columns. Then divide by 16.

In [ ]:
X = ...
print('Feature array:', X.shape)
print('Scaled pixel range:', X.min(), 'to', X.max())

In [ ]:
grader.check("q1")

### Q2. Make a 60 / 20 / 20 split

As in Lab 05, each set has a separate job:
- **Training:** fit the model's weights and biases.
- **Validation:** compare hidden-layer sizes and monitor learning.
- **Test:** evaluate the selected model once, at the end.

First hold out **20% of all images** as test data. Then hold out **25% of the remaining development data** as validation data. This leaves roughly 60% training, 20% validation, and 20% test data. Complete both calls below, using **X** and **y** in the first call.

**Hint:** `train_test_split(features, labels, test_size=..., random_state=SEED, stratify=labels)`. `stratify` keeps the digit proportions similar across the sets.

The provided lines convert the arrays into PyTorch **tensors**, arrays used for the network's calculations. Features use decimal values (`float32`); labels use integers (`long`). Dividing by the known pixel maximum of 16 does not estimate any statistics from the test set.

In [ ]:
X_dev, X_test, y_dev, y_test = train_test_split(
    ..., ..., test_size=..., random_state=SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=..., random_state=SEED, stratify=y_dev)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_val_t = torch.tensor(y_val, dtype=torch.long)

print('Train:', len(y_train), '| Validation:', len(y_val), '| Test:', len(y_test))

In [ ]:
grader.check("q2")

## Part 2: Build and train an MLP

An MLP learns intermediate features in a **hidden layer**, then uses those features to predict a class. Our network has the following structure:

**64 pixel values → 32 hidden units → 10 digit scores**

- `nn.Linear(a, b)` learns a weighted sum plus a bias for each of its `b` outputs.
- `nn.ReLU()` replaces negative values with zero. This nonlinearity lets the network learn more than a single linear transformation.
- The final outputs are **logits**, unrestricted scores for the ten classes. They are not probabilities. The largest score determines the predicted digit.

### Q3. Define the network

Complete **make_mlp** with two linear layers and a ReLU between them. Use **hidden_size** for the hidden-layer width so that we can change it later. The network should accept 64 input features and return 10 class scores.

**Hint:** Use `nn.Linear(input_size, output_size)` and `nn.ReLU()` inside `nn.Sequential`, which applies the layers in order. Do not add softmax; the loss function handles that during training.

In [ ]:
def make_mlp(hidden_size=32):
    torch.manual_seed(SEED)
    return nn.Sequential(
        ...,
        ...,
        ...,
    )

model = make_mlp(hidden_size=32)
print(model)
print('Output shape for five images:', model(X_train_t[:5]).shape)

In [ ]:
grader.check("q3")

### Q4. Complete the training loop

Training adjusts the weights and biases to reduce a **loss** measuring prediction error. Each **epoch** is one pass through the training set. Within an epoch, the model processes smaller groups of images called **batches**.

We use **cross-entropy loss** for classification and **Adam**, an optimizer that updates the parameters using gradients. PyTorch provides both algorithms. Cross-entropy takes the raw class scores and the correct integer labels.

Complete three lines in **train_model**:
- **criterion**: create the cross-entropy loss function.
- **logits**: compute the model's scores for **batch_X**.
- **loss**: compare those scores with **batch_y**.

**Hints:** Use `nn.CrossEntropyLoss()`, `model(batch_X)`, and `criterion(logits, batch_y)`.

The remaining steps are provided. `loss.backward()` calculates gradients, and `optimizer.step()` updates the parameters. `model.eval()` and `torch.no_grad()` prepare the validation calculation without tracking gradients. Run the completed cell to train a fresh 32-unit model for **30 epochs**.

In [ ]:
def train_model(model, epochs=30, learning_rate=0.001):
    criterion = ...
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    generator = torch.Generator().manual_seed(SEED)
    loader = DataLoader(TensorDataset(X_train_t, y_train_t),
                        batch_size=64, shuffle=True, generator=generator)
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for batch_X, batch_y in loader:
            optimizer.zero_grad()           # Clear the previous gradients.
            logits = ...
            loss = ...
            loss.backward()                 # Calculate gradients.
            optimizer.step()                # Update weights and biases.
            total_loss += loss.item() * len(batch_y)

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t)
            val_loss = criterion(val_logits, y_val_t).item()
            val_acc = (val_logits.argmax(dim=1) == y_val_t).float().mean().item()
        history['train_loss'].append(total_loss / len(y_train_t))
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch + 1:2d} | training loss: '
                  f"{history['train_loss'][-1]:.3f} | validation accuracy: {val_acc:.3f}")
    return history

# Re-create the model so re-running this cell starts training from scratch.
model = make_mlp(hidden_size=32)
history = train_model(model)

In [ ]:
grader.check("q4")

### Examine the learning curves

Run the next cell after Q4. It plots training and validation loss, along with validation accuracy. Lower loss indicates smaller prediction errors; higher accuracy means a larger share of correct predictions.

In [ ]:
epoch_numbers = np.arange(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epoch_numbers, history['train_loss'], label='Training')
axes[0].plot(epoch_numbers, history['val_loss'], label='Validation')
axes[0].set(xlabel='Epoch', ylabel='Cross-entropy loss', title='Is the loss going down?')
axes[0].legend()
axes[1].plot(epoch_numbers, history['val_acc'])
axes[1].set(xlabel='Epoch', ylabel='Accuracy', title='Validation accuracy', ylim=(0, 1))
plt.tight_layout()
plt.show()

**Your response:** Describe one trend in the loss plots and one in validation accuracy. If training loss continued falling while validation loss started rising, what would that suggest?

> Write your answer here.

## Part 3: Compare hidden-layer sizes

### Q5. Choose a hidden-layer width

Compare networks with **8, 32, and 64 hidden units**, using the same learning rate and number of epochs. The code reuses the 32-unit model from Q4 and trains the other two models. Complete the model creation and validation accuracy calculation.

**Hints:** Create each new network with `make_mlp(hidden_size=width)`. Compare **predictions** with **y_val_t**, then use `.float().mean().item()` to compute the share of correct predictions.

The provided code records training accuracy, validation accuracy, and the number of parameters. It selects the width with the highest validation accuracy, choosing the smaller model if scores tie. Do not use the test set to choose a width.

In [ ]:
candidates = {32: model}
rows = []
for width in [8, 32, 64]:
    if width != 32:
        candidate = ...
        train_model(candidate)
        candidates[width] = candidate
    candidate = candidates[width]
    candidate.eval()
    with torch.no_grad():
        train_pred = candidate(X_train_t).argmax(dim=1)
        predictions = candidate(X_val_t).argmax(dim=1)
        val_acc = ...
    rows.append({'hidden_units': width,
                 'parameters': sum(p.numel() for p in candidate.parameters()),
                 'train_acc': (train_pred == y_train_t).float().mean().item(),
                 'val_acc': val_acc})

width_results = pd.DataFrame(rows)
display(width_results.round(3))

# Highest validation accuracy wins; prefer the smaller model on an exact tie.
best_width = int(width_results.sort_values(
    ['val_acc', 'hidden_units'], ascending=[False, True]).iloc[0]['hidden_units'])
best_model = candidates[best_width]
print('Selected using validation data:', best_width, 'hidden units')

In [ ]:
grader.check("q5")

**Your response:** Which width was selected? Compare its training and validation accuracy with the other widths. Does the widest model have the best validation accuracy? Use the table to explain why training accuracy alone is not enough to choose a model.

> Write your answer here.

## Part 4: Evaluate the classifier

### Q6. Fit baselines and evaluate on test data

Create a **LogisticRegression** model named **baseline** with `max_iter=1000` and fit it on the training rows (**X_train**, **y_train**). Save the most common training digit as **majority_digit** for a baseline that always predicts that digit.

**Hints:** Create with `LogisticRegression(max_iter=1000)` and train with `.fit(X_train, y_train)`. `np.bincount(y_train).argmax()` gives the most common training label.

In [ ]:
baseline = ...
baseline.fit(..., ...)
majority_digit = ...

print('Logistic regression validation accuracy:', round(baseline.score(X_val, y_val), 3))
print('Majority digit:', majority_digit)

### Predict the test labels

The selected MLP is saved as **best_model**. Its width is now fixed. Compute its predictions for **X_test_t** as **test_pred**, then save its test accuracy as **test_acc**.

**Hints:** Call `best_model(X_test_t)`, then use `.argmax(dim=1).numpy()` to select the highest-scoring digit in each row and convert the results to a NumPy array. Use `accuracy_score(y_test, test_pred)` for accuracy. The evaluation settings are provided.

The table compares the selected MLP with both baselines. Report the measured results; a neural network is not guaranteed to outperform logistic regression on this dataset. Keep the models fixed after checking the test set so that the test scores remain a fair final evaluation.

In [ ]:
best_model.eval()
with torch.no_grad():
    test_pred = ...
test_acc = ...

test_results = pd.DataFrame({
    'model': ['Majority class', 'Logistic regression', f'MLP ({best_width} hidden units)'],
    'test_accuracy': [np.mean(y_test == majority_digit),
                      baseline.score(X_test, y_test), test_acc],
})
display(test_results.round(3))

In [ ]:
grader.check("q6")

**Your response:** Report the MLP's test accuracy as a percentage. Did it outperform the majority-class baseline and logistic regression?

> Write your answer here.

### Q7. Interpret classification errors

Run the next cell after Q6. In the **confusion matrix**, rows are true digits and columns are predicted digits. Entries off the diagonal are mistakes. The images below the matrix show up to eight incorrect predictions.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(y_test, test_pred, labels=np.arange(10),
                                       cmap='Blues', colorbar=False, ax=ax)
ax.set_title('Selected MLP: test confusion matrix')
plt.tight_layout()
plt.show()

mistakes = np.flatnonzero(test_pred != y_test)
if len(mistakes) == 0:
    print('No mistakes on this split. That does not guarantee perfect future performance!')
else:
    chosen = mistakes[:8]
    fig, axes = plt.subplots(1, len(chosen), figsize=(2 * len(chosen), 2.5), squeeze=False)
    for ax, i in zip(axes.flat, chosen):
        ax.imshow(X_test[i].reshape(8, 8), cmap='gray', vmin=0, vmax=1)
        ax.set_title(f'True: {y_test[i]}\nPred: {test_pred[i]}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

**Your response:** (a) Choose one incorrect prediction and describe why the two digits may be confused. (b) Would you expect the same accuracy on phone photos with different lighting and backgrounds? Give one reason.

> Write your answer here.

In [ ]:
grader.check_all()

## Wrap-up

You prepared image features, trained an MLP, selected a hidden-layer size using validation data, and evaluated its predictions on test data.

**Before submitting:** complete the coding exercises in Q1–Q6 and the written responses through Q7. Restart the kernel and run the main lab from top to bottom. Keep the plots and outputs in the completed notebook.

The sections below are optional extensions.

## Optional extension: Learned embeddings

An **embedding** represents an item as a vector. The selected MLP maps each image's 64 pixel values to a hidden-layer vector, with one feature per hidden unit. These features are learned to help classify the digits.

Run the provided cell to extract the embeddings and plot them in two dimensions using **PCA from Lab 06**. PCA is fit on the training embeddings and applied to the validation embeddings. No code changes are needed.

In [ ]:
best_model.eval()
with torch.no_grad():
    train_embeddings = best_model[1](best_model[0](X_train_t)).numpy()
    val_embeddings = best_model[1](best_model[0](X_val_t)).numpy()

pca = PCA(n_components=2).fit(train_embeddings)
embedding_2d = pca.transform(val_embeddings)
print('Pixels:', X_val.shape, '→ Embeddings:', val_embeddings.shape,
      '→ Plot:', embedding_2d.shape)

fig, ax = plt.subplots(figsize=(8, 5))
points = ax.scatter(embedding_2d[:, 0], embedding_2d[:, 1], c=y_val,
                    cmap='tab10', s=24, alpha=0.75, vmin=-0.5, vmax=9.5)
fig.colorbar(points, ax=ax, ticks=range(10), label='True digit')
ax.set(xlabel='PC1', ylabel='PC2', title='Validation images: learned embeddings in 2D')
plt.tight_layout()
plt.show()
print('Variance kept in the plot:', round(pca.explained_variance_ratio_.sum(), 3))

**Your response (optional):** Describe any digit groups visible in the plot. Why might two digits overlap in two dimensions even if the classifier can separate them using the full embedding?

## Optional extension: NLP and transformers

Digit recognition is a **computer vision** task: the model learns from images. In **natural language processing (NLP)**, the inputs are text. A **tokenizer** divides text into tokens, often words or word pieces, and an embedding layer maps each token ID to a vector.

In a **transformer**, attention lets each token combine information from other tokens. For example, the meaning of “bank” depends on the surrounding words in “river bank” and “bank account.” Transformers also contain MLPs that transform the token features.

For images, **convolutional neural networks (CNNs)** use shared filters over local neighborhoods. **Vision transformers** process image patches using position information and attention. These approaches use image structure differently from the MLP in this lab.

**Your response (optional):** (a) How does a hidden-layer embedding differ from the final predicted class? (b) Why might the representation of “bank” need to depend on the surrounding words?

### Further reading

- [Digits dataset documentation](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html)
- [PyTorch: build a neural network](https://docs.pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html)
- [PyTorch: optimization and training](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html)
- [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/) — optional visual reading